# Search evaluation

Evaluate article-level and chunk-level retrieval using realistic Khmer queries and manually verified expected article IDs. The notebook reports the actual retrieved results and Recall@K, Hit@K, MRR, and nDCG@K.

In [1]:
import gc
import json
import logging
import math
import unicodedata
from pathlib import Path

import chromadb
import numpy as np

logging.getLogger('torch.distributed.elastic.multiprocessing.redirects').setLevel(logging.ERROR)
try:
    from transformers.utils import logging as transformers_logging
    transformers_logging.disable_progress_bar()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'data-pipeline').is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / 'data-pipeline' / 'data'
CHROMA_DIR = DATA_DIR / 'processed' / 'chroma_db'
QUERY_PATH = PROJECT_ROOT / 'data-pipeline' / 'evaluation' / 'search_queries.json'
QUERIES = json.loads(QUERY_PATH.read_text(encoding='utf-8'))
TOP_K = 5
MODELS = {
    'sealion': {'model': 'aisingapore/SEA-LION-E5-Embedding-600M', 'article_collection': 'legal_articles_sealion', 'chunk_collection': 'legal_chunks_sealion', 'sentence_chunk_collection': 'legal_sentence_chunks_sealion'},
    'granite-r2': {'model': 'ibm-granite/granite-embedding-311m-multilingual-r2', 'article_collection': 'legal_articles_granite_r2', 'chunk_collection': 'legal_chunks_granite_r2', 'sentence_chunk_collection': 'legal_sentence_chunks_granite_r2'},
}
MODEL_CACHE = {}

def normalize_query(query):
    text = unicodedata.normalize('NFC', query)
    text = ''.join(char for char in text if char not in '\u200b\u200c\u200d\ufeff')
    return ' '.join(text.replace('\u00a0', ' ').split())

def embed_query(query, model_key):
    from sentence_transformers import SentenceTransformer
    if model_key not in MODEL_CACHE:
        MODEL_CACHE[model_key] = SentenceTransformer(MODELS[model_key]['model'])
    vector = MODEL_CACHE[model_key].encode([query], show_progress_bar=False, convert_to_numpy=True, normalize_embeddings=True)[0]
    return np.asarray(vector, dtype='float32')

def search(query, model_key, level, top_k=TOP_K):
    client = chromadb.PersistentClient(path=str(CHROMA_DIR))
    collection = client.get_collection(MODELS[model_key][f'{level}_collection'])
    result = collection.query(query_embeddings=[embed_query(normalize_query(query), model_key).tolist()], n_results=top_k, include=['metadatas', 'distances'])
    rows = []
    for rank, item_id in enumerate(result['ids'][0]):
        metadata = result['metadatas'][0][rank] or {}
        article_id = f"{metadata.get('document_id')}-{metadata.get('article_number')}" if metadata.get('document_id') is not None and metadata.get('article_number') is not None else metadata.get('article_id', item_id)
        rows.append({'rank': rank + 1, 'id': item_id, 'article_id': article_id, 'distance': result['distances'][0][rank]})
    return rows

def retrieved_article_ids(rows, level):
    return [row['article_id'] if level.endswith('chunk') else row['id'] for row in rows]

def metrics(retrieved, expected, k=TOP_K):
    expected = set(expected)
    top = retrieved[:k]
    hits = [rank for rank, item in enumerate(top, start=1) if item in expected]
    first_rank = hits[0] if hits else None
    hit_at_k = 1.0 if hits else 0.0
    recall_at_k = len(set(top) & expected) / len(expected)
    mrr = 1.0 / first_rank if first_rank else 0.0
    dcg = sum(1.0 / math.log2(rank + 1) for rank, item in enumerate(top, start=1) if item in expected)
    ideal_hits = min(len(expected), k)
    idcg = sum(1.0 / math.log2(rank + 1) for rank in range(1, ideal_hits + 1))
    return {'hit_at_5': hit_at_k, 'recall_at_5': recall_at_k, 'mrr': mrr, 'ndcg_at_5': dcg / idcg if idcg else 0.0}

print(f'Queries: {len(QUERIES)}')
print('Expected IDs are stored in search_queries.json and must be reviewed before treating scores as final.')

Queries: 20
Expected IDs are stored in search_queries.json and must be reviewed before treating scores as final.


In [2]:
all_results = []
for model_key in MODELS:
    for level in ('article', 'chunk', 'sentence_chunk'):
        for item in QUERIES:
            rows = search(item['query'], model_key, level)
            retrieved = retrieved_article_ids(rows, level)
            all_results.append({'query_id': item['query_id'], 'query': item['query'], 'model': model_key, 'level': level, 'expected': item['expected_article_ids'], 'actual_top_5': rows, 'metrics': metrics(retrieved, item['expected_article_ids'])})

summary = {}
for result in all_results:
    key = f"{result['level']}_{result['model']}"
    summary.setdefault(key, []).append(result['metrics'])
summary = {key: {metric: sum(row[metric] for row in rows) / len(rows) for metric in rows[0]} for key, rows in summary.items()}

print(json.dumps({'summary': summary, 'results': all_results}, ensure_ascii=False, indent=2))

{
  "summary": {
    "article_sealion": {
      "hit_at_5": 0.95,
      "recall_at_5": 0.925,
      "mrr": 0.775,
      "ndcg_at_5": 0.8030357986621943
    },
    "chunk_sealion": {
      "hit_at_5": 0.9,
      "recall_at_5": 0.875,
      "mrr": 0.7625,
      "ndcg_at_5": 0.8323910987988248
    },
    "sentence_chunk_sealion": {
      "hit_at_5": 0.9,
      "recall_at_5": 0.875,
      "mrr": 0.7625,
      "ndcg_at_5": 0.8323910987988248
    },
    "article_granite-r2": {
      "hit_at_5": 0.85,
      "recall_at_5": 0.825,
      "mrr": 0.6366666666666666,
      "ndcg_at_5": 0.6696394630357186
    },
    "chunk_granite-r2": {
      "hit_at_5": 0.8,
      "recall_at_5": 0.775,
      "mrr": 0.62,
      "ndcg_at_5": 0.6977197786179612
    },
    "sentence_chunk_granite-r2": {
      "hit_at_5": 0.75,
      "recall_at_5": 0.725,
      "mrr": 0.61,
      "ndcg_at_5": 0.6797180366620574
    }
  },
  "results": [
    {
      "query_id": "q01",
      "query": "តើការបើកបរយានយន្តនៅពេលមានជាតិស្រវឹងត